# Ch. 5 · TP Assistant scientifique traçable (mini-RAG)

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/benlahmar/IASciences/blob/main/notebooks/ch05-mini-rag.ipynb)

**IA pour les sciences — M1.** Recherche TF-IDF avec extraits vérifiables, évaluation de la récupération et refus de répondre — sans LLM externe.

**Durée indicative :** 3 h

**Objectifs**

- Construire un corpus documenté et identifié.
- Comprendre TF-IDF et la similarité cosinus en calculant un exemple à la main.
- Évaluer un moteur de recherche avec précision@k et rappel@k sur des questions annotées.
- Produire une réponse extractive citée et savoir répondre « information absente ».

**Prérequis :** cours du chapitre 5 (embeddings, similarité, RAG).

> **Mode d’emploi.** Exécutez les cellules dans l’ordre. Les cellules marquées **✏️ Exercice** contiennent des `...` à compléter : une cellule non complétée lève une erreur, c’est normal. Les corrigés sont repliés sous **« Voir le corrigé »** : cherchez d’abord ! Les cellules suivantes ne dépendent jamais du résultat d’un exercice.

## 1. Un corpus traçable

Chaque passage a un **identifiant** et une **source** : sans eux, aucune citation n’est vérifiable.

In [ ]:
MOTS_VIDES = ('a au aux avec ce ces cette comment d de des du dans elle en est et il ils l la le les leur lors lorsque '
              'ne ni on ou par pas pour qu que quel quelle quels quelles qui sa se ses son sont sous sur un une '
              'faut doit peut quoi').split()
documents = [
 {"id": "BIO-01", "source": "Note de laboratoire biologie",
  "texte": "Une classification d'espèces doit être évaluée sur des échantillons indépendants, prélevés sur des individus différents de ceux de l'entraînement."},
 {"id": "BIO-02", "source": "Protocole de culture cellulaire",
  "texte": "Les cultures sont incubées à 37 °C sous 5 % de CO2 ; un effet de lot apparaît lorsque les plaques proviennent de séries différentes."},
 {"id": "CHIM-01", "source": "Note de laboratoire chimie",
  "texte": "La solubilité dépend notamment de la température et du solvant ; elle est mesurée en mg/L à 25 °C."},
 {"id": "CHIM-02", "source": "Fiche de spectroscopie",
  "texte": "Un spectre infrarouge présente des bandes d'absorption caractéristiques ; la ligne de base doit être corrigée avant comparaison."},
 {"id": "PHY-01", "source": "Note de laboratoire physique",
  "texte": "Les mesures répétées permettent d'estimer la dispersion expérimentale et l'incertitude de type A."},
 {"id": "PHY-02", "source": "Cahier d'expérience thermique",
  "texte": "La température décroît de façon exponentielle lors du refroidissement ; la constante de temps est estimée par ajustement."},
 {"id": "GEO-01", "source": "Note de terrain géologie",
  "texte": "Une carte géologique dépend de l'échelle d'observation ; les sites voisins sont spatialement corrélés."},
 {"id": "GEO-02", "source": "Rapport géochimique",
  "texte": "Les teneurs en silice et en magnésium distinguent les basaltes des granites dans les échantillons analysés."},
 {"id": "STAT-01", "source": "Guide de validation",
  "texte": "La validation croisée par groupe évite qu'un même patient ou un même site apparaisse à la fois en entraînement et en test."},
 {"id": "STAT-02", "source": "Guide de validation",
  "texte": "Un intervalle de confiance par bootstrap quantifie l'incertitude d'une métrique calculée sur un petit jeu de test."},
]

print(len(documents), 'passages')
for d in documents[:3]:
    print(d['id'], '—', d['texte'][:70], '…')

## 2. TF-IDF, à la main puis avec scikit-learn

Pour un terme $t$ et un document $d$ :
$\text{tf}(t,d)$ = nombre d’occurrences ; $\text{idf}(t) = \ln\frac{1+N}{1+\text{df}(t)} + 1$ (formule de scikit-learn),
où $\text{df}(t)$ est le nombre de documents contenant $t$. Un terme présent partout (« la », « de ») a un IDF faible — on retire même ces *mots vides* ;
un terme rare (« bootstrap ») a un IDF élevé. Chaque vecteur est ensuite normalisé, et la similarité est le **cosinus**
$\cos(a,b)=\frac{a\cdot b}{\|a\|\|b\|}$.

In [ ]:
import numpy as np
N = len(documents)
for terme in ['validation', 'bootstrap', 'température']:
    df_t = sum(terme in d['texte'].lower() for d in documents)
    idf = np.log((1 + N) / (1 + df_t)) + 1
    print(f'{terme:12s} présent dans {df_t} document(s) -> idf = {idf:.3f}')

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

textes = [d['texte'] for d in documents]
vec = TfidfVectorizer(ngram_range=(1, 2), strip_accents='unicode', lowercase=True, stop_words=MOTS_VIDES)
X = vec.fit_transform(textes)
print('Taille du vocabulaire (unigrammes + bigrammes) :', len(vec.vocabulary_))

def rechercher(question, k=3, vectoriseur=vec, matrice=X):
    scores = cosine_similarity(vectoriseur.transform([question]), matrice).ravel()
    indices = scores.argsort()[::-1][:k]
    return [(documents[i], float(scores[i])) for i in indices if scores[i] > 0]

for doc, score in rechercher('comment évaluer une classification ?'):
    print(f'{doc["id"]:8s} {score:.3f}  {doc["texte"][:80]}')

## 3. Évaluer la récupération

Un moteur se juge sur des **questions annotées** : pour chacune, on indique quels passages sont pertinents.
- $\text{précision@}k$ = part des $k$ passages récupérés qui sont pertinents ;
- $\text{rappel@}k$ = part des passages pertinents retrouvés dans les $k$ premiers.

In [ ]:
questions = {
    "Comment éviter qu'un même patient soit en entraînement et en test ?": {'STAT-01', 'BIO-01'},
    "Comment estimer l'incertitude d'une métrique ?": {'STAT-02', 'PHY-01'},
    "De quoi dépend la solubilité ?": {'CHIM-01'},
    "Qu'est-ce qu'un effet de lot ?": {'BIO-02'},
    "Comment distinguer basalte et granite ?": {'GEO-02'},
    "Comment se refroidit un objet chaud ?": {'PHY-02'},
}

def evaluer_recherche(k=3, **kw):
    P, R = [], []
    for q, pertinents in questions.items():
        trouves = {d['id'] for d, s in rechercher(q, k=k, **kw)}
        P.append(len(trouves & pertinents) / k)
        R.append(len(trouves & pertinents) / len(pertinents))
    return np.mean(P), np.mean(R)

for k in [1, 2, 3]:
    p, r = evaluer_recherche(k)
    print(f'k={k} : précision@k = {p:.2f}   rappel@k = {r:.2f}')

**🔎 À observer.** Deux questions échouent complètement. « Comment se refroidit un objet chaud ? » ne partage aucun mot avec PHY-02
(« température décroît… refroidissement ») : **synonymes et dérivés**. « basalte et granite » ne trouve pas « basaltes… granites » : **pluriels**.
La recherche **lexicale** compare des chaînes de caractères, pas des sens. C’est la motivation de la lemmatisation et surtout des **embeddings** neuronaux.

## 4. Répondre sans inventer : réponse extractive et seuil de refus

La réponse n’est composée **que** de passages du corpus, chacun suivi de son identifiant. Sous un seuil de similarité,
on répond « information absente du corpus » plutôt que de proposer un passage sans rapport.

In [ ]:
def reponse_extractive(question, k=2, seuil=0.10):
    passages = [(d, s) for d, s in rechercher(question, k=k) if s >= seuil]
    if not passages:
        return 'Information absente du corpus.'
    return '\n'.join(f'- {d["texte"]} [{d["id"]}, score {s:.2f}]' for d, s in passages)

for q in ['comment évaluer une classification ?', 'quelle est la dose recommandée de paracétamol ?']:
    print('Q :', q); print(reponse_extractive(q)); print()

### ✏️ Exercice 1 — Choisir le seuil de refus

Ajoutez trois questions **hors corpus** (`hors_corpus`). Pour des seuils de 0 à 0,4, comptez (a) les questions hors corpus correctement refusées et (b) les questions de `questions` qui reçoivent au moins un passage pertinent. Quel seuil retenez-vous ?

In [ ]:
hors_corpus = ['Quelle est la dose recommandée de paracétamol ?',
               'Qui a gagné la coupe du monde 1998 ?',
               'Comment programmer un microcontrôleur ?']
for seuil in [0, 0.05, 0.1, 0.2, 0.3, 0.4]:
    refus = sum(... for q in hors_corpus)
    bonnes = sum(... for q, pert in questions.items())
    print(f'seuil {seuil:.2f} : refus corrects {refus}/3, questions servies {bonnes}/{len(questions)}')

<details><summary><b>Voir le corrigé</b></summary>

```python
hors_corpus = ['Quelle est la dose recommandée de paracétamol ?',
               'Qui a gagné la coupe du monde 1998 ?',
               'Comment programmer un microcontrôleur ?']
for seuil in [0, 0.05, 0.1, 0.2, 0.3, 0.4]:
    refus = sum(all(s < seuil for d, s in rechercher(q, k=2)) for q in hors_corpus)
    bonnes = sum(any(d['id'] in pert and s >= seuil for d, s in rechercher(q, k=2)) for q, pert in questions.items())
    print(f'seuil {seuil:.2f} : refus corrects {refus}/3, questions servies {bonnes}/{len(questions)}')
```

Le seuil arbitre entre **refuser à tort** et **répondre à tort**. Grâce aux mots vides retirés, les questions hors corpus ont un score nul ; mais un seuil trop élevé abandonne de vraies questions. Ce choix doit être fait sur des questions de validation, puis vérifié sur de nouvelles questions.

</details>

### ✏️ Exercice 2 — Unigrammes ou bigrammes ?

Construisez un second vectoriseur avec `ngram_range=(1, 1)` et comparez précision@3 et rappel@3 avec `evaluer_recherche(3, vectoriseur=..., matrice=...)`.

In [ ]:
vec1 = TfidfVectorizer(ngram_range=(1, 1), strip_accents='unicode', stop_words=MOTS_VIDES)
X1 = ...
print('bigrammes  :', evaluer_recherche(3))
print('unigrammes :', evaluer_recherche(3, vectoriseur=..., matrice=...))

<details><summary><b>Voir le corrigé</b></summary>

```python
vec1 = TfidfVectorizer(ngram_range=(1, 1), strip_accents='unicode', stop_words=MOTS_VIDES)
X1 = vec1.fit_transform(textes)
print('bigrammes  :', evaluer_recherche(3))
print('unigrammes :', evaluer_recherche(3, vectoriseur=vec1, matrice=X1))
```

Ici les deux variantes font jeu égal : sur un corpus aussi petit, les différences dépendent de quelques questions. Il faut un jeu de questions plus grand pour conclure.

</details>

## 5. Et le LLM ?

Ce TP s’arrête volontairement avant la génération. Le notebook facultatif **`ch05-atelier-llm.ipynb`** ajoute un LLM
derrière cette recherche et montre comment **vérifier automatiquement** les citations et les nombres de sa réponse.

## Interprétation scientifique (à rédiger)

1. Quelles questions le moteur lexical échoue-t-il à servir, et pourquoi ?
2. Quel seuil de refus avez-vous retenu, avec quel compromis ?
3. En quoi une réponse extractive citée est-elle plus vérifiable qu’une réponse générée ?
4. Que faudrait-il pour évaluer ce système sur un vrai corpus d’articles de votre discipline ?